<a href="https://colab.research.google.com/github/mayank261193/Session-4/blob/main/Session4_FollowAlong.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 4 — OOP Basics, Tooling & First Libraries
### For Policy, Governance & UPSC Learners

In Sessions 1–3 you learned Python's grammar: variables, data types, conditionals, loops, functions, and the core containers (lists, tuples, dictionaries, sets).

Today you move from *writing scripts* to *thinking like a policy data analyst*:
- **Object-Oriented Programming (OOP)** — organising code around real "things" like a `GovernmentScheme`, a `DistrictData`, an `ElectionResult`.
- **Professional tooling** — `pip`, virtual environments, Jupyter, Git/GitHub.
- **Your first data libraries** — **NumPy** and **Pandas**, used to crunch census, budget, and scheme data.

> **Why this matters for governance work:** every scheme dashboard, every census table, every budget analysis you will build in Python runs on NumPy and Pandas, and lives in a Git repository.

### What you'll be able to do by the end of today
1. Explain what OOP is and why it exists
2. Create Python classes and objects
3. Use `__init__()` and object attributes
4. Write methods (behaviours) on your classes
5. Refactor dictionary-based scheme code into a clean class
6. Install packages with `pip`; understand virtual environments
7. Work confidently in Jupyter Notebook
8. Understand Git and GitHub
9. Use **NumPy** arrays for fast maths on numeric data
10. Use **Pandas** DataFrames to load and explore a district CSV

**Icons you'll see:** Instructor Tip · Common Mistake · Key Takeaway · Checkpoint · Mini Exercise · Prediction Question · Debugging Exercise · Challenge.

*(Detailed theory lives in the PPT slides — this notebook is code-first. Run every cell yourself.)*

---

## 2. Why Object-Oriented Programming?

**What it is:** Until now your code was *procedural* — data (variables, dictionaries) in one place, actions (functions) in another. **OOP** bundles data *and* the actions on it into one unit called an **object**.
**Why it matters:** As governance data grows (500 schemes, not 2), keeping data and its logic together keeps code consistent and reusable.

### Let's feel the pain first — schemes as loose dictionaries

In [ ]:
# Procedural style: scheme data (dicts) sits apart from the functions acting on it.
scheme_one = {"name": "PM-KISAN", "ministry": "Agriculture", "beneficiaries_lakh": 1100}
scheme_two = {"name": "PMAY-G", "ministry": "Rural Development", "beneficiaries_lakh": 290}

def describe_scheme(scheme):
    print(scheme["name"], "run by", scheme["ministry"], "covers",
          scheme["beneficiaries_lakh"], "lakh beneficiaries")

describe_scheme(scheme_one)
describe_scheme(scheme_two)

PM-KISAN run by Agriculture covers 1100 lakh beneficiaries
PMAY-G run by Rural Development covers 290 lakh beneficiaries


### Output discussion
Fine for two schemes. But with hundreds of schemes and many functions (`describe`, `revise_budget`, `check_coverage`), every function must remember the exact keys — misspell `"ministry"` and Python crashes later. The **data** and its **behaviour** are drifting apart.

> **The problem OOP solves:** bundle data + its actions into one self-contained unit so they stay together and consistent.

**Key Takeaway:** OOP keeps a scheme's data and its behaviour in one object instead of scattering them across your file.

---

## 3. Classes and Objects

**What it is:** A **class** is a *blueprint*; an **object** is a *thing built from it*. `class GovernmentScheme` is the blueprint (written once); each real scheme you create from it is an **object** (an **instance**).
**Why it matters:** One blueprint, many schemes — write the structure once, reuse it endlessly.

**Instructor Tip:** A class is the policy *template*; an object is one *filled-in* scheme.

In [ ]:
# The simplest class: a blueprint with nothing in it yet. 'pass' = intentionally empty.
class GovernmentScheme:
    pass

# Build two objects (instances) from the blueprint.
first_scheme = GovernmentScheme()
second_scheme = GovernmentScheme()

print(first_scheme)
print(second_scheme)
print(type(first_scheme))

<class '__main__.GovernmentScheme'>


### Output discussion
You saw something like `<__main__.GovernmentScheme object at 0x...>`. The `0x...` is a **memory address** — where that object lives. The two schemes printed *different* addresses: proof they are **two separate objects** from the same blueprint.

**Common Mistake:** `GovernmentScheme` without `()` just renames the blueprint — you almost always want `GovernmentScheme()` **with** parentheses to build an object.

### Mini Exercise 1
Create an empty class `DistrictData` (use `pass`). Build two district objects `pune` and `patna`, print both, and note whether their addresses differ.

In [ ]:
class DistrictData:
    pass

pune = DistrictData()
patna = DistrictData()

print(pune)
print(patna)
# The two addresses are DIFFERENT — pune and patna are separate objects
# built from the same DistrictData blueprint.

**Checkpoint — Classes & Objects:** A class is a blueprint written once; an object is a real thing built with `ClassName()` (parentheses required); two objects from one class are still separate.
**Key Takeaway:** One class (blueprint), many objects — always build with `ClassName()`.

---

## 4. Constructors — the `__init__()` Method

**What it is:** `__init__` (say "dunder init") is a special function that runs **automatically** when you create an object, setting up its starting data. `self` means *this particular object* being built.
**Why it matters:** It lets each scheme carry its own name, ministry, and budget from the moment it is created.

**Instructor Tip:** You never pass `self` yourself — Python fills it in. You write `GovernmentScheme("PM-KISAN", "Agriculture")`.

In [ ]:
class GovernmentScheme:
    # __init__ runs automatically when a scheme is created.
    # 'self' is this particular scheme; name and ministry are passed in.
    def __init__(self, name, ministry):
        self.name = name          # attach a 'name' attribute to THIS scheme
        self.ministry = ministry  # attach a 'ministry' attribute

pmkisan = GovernmentScheme("PM-KISAN", "Agriculture")
pmay = GovernmentScheme("PMAY-G", "Rural Development")

# Read attributes back with dot notation.
print(pmkisan.name, "->", pmkisan.ministry)
print(pmay.name, "->", pmay.ministry)

PM-KISAN -> Agriculture
PMAY-G -> Rural Development


### Output discussion
`GovernmentScheme("PM-KISAN", "Agriculture")` built a new object and ran `__init__` with `self` = that object. `self.name = name` stored the value **on** the object as an **attribute**, which `pmkisan.name` reads back. Each scheme holds its own separate data.

### Attributes can change after creation

In [ ]:
# Attributes are just data on the object — you can update them (e.g. a budget revision).
pmkisan.budget_crore = 60000
print("Before revision:", pmkisan.budget_crore)

pmkisan.budget_crore = 63500   # Budget revised at supplementary demand
print("After revision :", pmkisan.budget_crore)

Before revision: 60000
After revision : 63500


### Debugging Exercise 1 — the classic missing `self`
This crashes. The first parameter of `__init__` must be `self`.

```python
class GovernmentScheme:
    def __init__(name, ministry):   # missing self!
        self.name = name
        self.ministry = ministry

s = GovernmentScheme("MGNREGA", "Rural Development")   # TypeError
```
Without `self` as the first parameter, the new object gets stuffed into `name`, arguments shift, and it fails.

In [1]:
# Fixed version — self is the first parameter.
class GovernmentScheme:
    def __init__(self, name, ministry):
        self.name = name
        self.ministry = ministry

s = GovernmentScheme("MGNREGA", "Rural Development")
print(s.name, s.ministry)

MGNREGA Rural Development


**Common Mistake:** Forgetting `self.` when storing (`name = name` instead of `self.name = name`) — the value vanishes when `__init__` ends.

### Mini Exercise 2
Create `GovernmentScheme` with a constructor taking `name`, `ministry`, **and** `budget_crore`. Build "Ayushman Bharat", ministry "Health & Family Welfare", budget `7200`. Print all three on one line.

In [ ]:
class GovernmentScheme:
    def __init__(self, name, ministry, budget_crore):
        self.name = name
        self.ministry = ministry
        self.budget_crore = budget_crore

ayushman = GovernmentScheme("Ayushman Bharat", "Health & Family Welfare", 7200)
print(ayushman.name, ayushman.ministry, ayushman.budget_crore)

Ayushman Bharat Health & Family Welfare 7200


### Prediction Question 1
Predict the output:

```python
class ElectionResult:
    def __init__(self, party):
        self.party = party

seat_one = ElectionResult("Party A")
seat_two = ElectionResult("Party B")
seat_one.party = "Party C"          # recount / re-poll
print(seat_one.party, seat_two.party)
```
**Answer:** `Party C Party B`. The two seats are separate objects — changing one never touches the other.

**Checkpoint — Constructors:** `__init__(self, ...)` runs automatically; `self` is "this object"; `self.x = x` creates an attribute; read it back with `object.x`.
**Key Takeaway:** `__init__` gives each new object its own data; `self` is always first.

---

## 5. Methods — Giving Objects Behaviour

**What it is:** A **method** is a function inside a class — something the object can *do*. An **attribute** is what an object *has* (name, budget); a **method** is what it *does* (describe, launch, review). A method sees the object's own data through `self`.
**Why it matters:** This is the core OOP payoff — data and behaviour living in one object.

In [ ]:
class GovernmentScheme:
    def __init__(self, name, ministry):
        self.name = name
        self.ministry = ministry

    # Methods take 'self' first, so they can read this scheme's own data.
    def describe(self):
        print(self.name, "is run by the Ministry of", self.ministry)

    def launch(self):
        print(self.name, "has been officially launched.")

    def review(self):
        print("Conducting annual review of", self.name)

pmkisan = GovernmentScheme("PM-KISAN", "Agriculture")

# Call methods with dot notation AND parentheses (they are actions).
pmkisan.describe()
pmkisan.launch()
pmkisan.review()

PM-KISAN is run by the Ministry of Agriculture
PM-KISAN has been officially launched.
Conducting annual review of PM-KISAN


### Output discussion
Each method used the scheme's own name via `self.name` — we never passed `"PM-KISAN"` in. **Common Mistake:** `pmkisan.describe` (no parentheses) just refers to the method; actions need their `()`.

In [ ]:
# See the difference:
print(pmkisan.describe)   # no () -> refers to the method, doesn't run it
pmkisan.describe()        # with () -> actually runs it

<bound method GovernmentScheme.describe of <__main__.GovernmentScheme object at 0x7e150d2d58b0>>
PM-KISAN is run by the Ministry of Agriculture


### A method that both uses data AND changes it — a Union Budget head

In [ ]:
class UnionBudget:
    def __init__(self, head, allocation_crore):
        self.head = head                       # e.g. "Education"
        self.allocation_crore = allocation_crore

    def spend(self, amount):
        self.allocation_crore = self.allocation_crore - amount   # update own data
        print("Spent", amount, "cr on", self.head, "-> remaining:", self.allocation_crore)

    def display(self):
        print("Budget head:", self.head, "| Remaining:", self.allocation_crore, "cr")

education = UnionBudget("Education", 1000)
education.display()
education.spend(350)
education.display()

Budget head: Education | Remaining: 1000 cr
Spent 350 cr on Education -> remaining: 650
Budget head: Education | Remaining: 650 cr


### Output discussion
`spend(350)` reached into the object, reduced `self.allocation_crore`, and stored it back — methods can *change* data, not just read it.

### Mini Exercise 3
Add an `allocate(self, amount)` method to `UnionBudget` that *adds* funds and prints the new total. Build a "Health" head with `800`, allocate `250`, and display it.

In [ ]:
class UnionBudget:
    def __init__(self, head, allocation_crore):
        self.head = head
        self.allocation_crore = allocation_crore

    def spend(self, amount):
        self.allocation_crore = self.allocation_crore - amount
        print("Spent", amount, "cr on", self.head, "-> remaining:", self.allocation_crore)

    def allocate(self, amount):
        self.allocation_crore = self.allocation_crore + amount
        print("Allocated", amount, "cr to", self.head, "-> total:", self.allocation_crore)

    def display(self):
        print("Budget head:", self.head, "| Total:", self.allocation_crore, "cr")

health = UnionBudget("Health", 800)
health.allocate(250)
health.display()

Allocated 250 cr to Health -> total: 1050
Budget head: Health | Total: 1050 cr


### Prediction Question 2
What does this print?

```python
class Minister:
    def __init__(self, name):
        self.name = name
    def take_oath(self):
        print(self.name, "takes the oath of office.")

m = Minister("Sitharaman")
m.take_oath()
```
**Answer:** `Sitharaman takes the oath of office.` — `self` becomes `m`, so `self.name` is `"Sitharaman"`.

**Checkpoint — Methods:** a method is behaviour inside a class; it takes `self` first; call it with `()`; it can read *and* change attributes.
**Key Takeaway:** Attributes are what an object *has*; methods are what it *does*; `self` bridges them.

---

## 6. Guided Practice — Building Classes Together

**What it is:** Repetition of the same `__init__` + `display()` shape until it feels automatic. We build one together, then you build three.

**Instructor Tip:** A `display()` method that prints an object's details is a habit worth building — you'll write dozens.

In [ ]:
class Minister:
    def __init__(self, name, portfolio):
        self.name = name
        self.portfolio = portfolio

    def display(self):
        print("Minister:", self.name, "| Portfolio:", self.portfolio)

finance = Minister("N. Sitharaman", "Finance")
defence = Minister("R. Singh", "Defence")

finance.display()
defence.display()

Minister: N. Sitharaman | Portfolio: Finance
Minister: R. Singh | Portfolio: Defence


### Output discussion
Same blueprint, two ministers, each with its own data shown via `self`. Now build the next three yourself before peeking.

### Mini Exercise 4 — an `ElectionResult` class
Attributes `constituency`, `winning_party`, `margin`, plus `display()`. Create *Varanasi*, *Party A*, margin *152000*, and display it.

In [ ]:
class ElectionResult:
    def __init__(self, constituency, winning_party, margin):
        self.constituency = constituency
        self.winning_party = winning_party
        self.margin = margin

    def display(self):
        print("Seat:", self.constituency, "| Won by:", self.winning_party,
              "| Margin:", self.margin)

varanasi = ElectionResult("Varanasi", "Party A", 152000)
varanasi.display()

Seat: Varanasi | Won by: Party A | Margin: 152000


### Mini Exercise 5 — a `CensusRecord` class
Attributes `state`, `population_lakh`, `literacy_rate`, plus `display()`. Create *Kerala*, `330`, `94`. Display it.

In [ ]:
class CensusRecord:
    def __init__(self, state, population_lakh, literacy_rate):
        self.state = state
        self.population_lakh = population_lakh
        self.literacy_rate = literacy_rate

    def display(self):
        print("Census:", self.state, "|", self.population_lakh, "lakh people |",
              self.literacy_rate, "% literacy")

kerala = CensusRecord("Kerala", 330, 94)
kerala.display()

Census: Kerala | 330 lakh people | 94 % literacy


### Mini Exercise 6 — a `DistrictData` class
Attributes `name`, `state`, `budget_crore`, plus `display()`. Create *Pune*, *Maharashtra*, `320`. Display it.

**Instructor Tip:** Keep this `DistrictData` class — near the end we load `DistrictData` objects into a Pandas DataFrame.

In [ ]:
class DistrictData:
    def __init__(self, name, state, budget_crore):
        self.name = name
        self.state = state
        self.budget_crore = budget_crore

    def display(self):
        print("District:", self.name, "| State:", self.state,
              "| Budget:", self.budget_crore, "cr")

pune = DistrictData("Pune", "Maharashtra", 320)
pune.display()

District: Pune | State: Maharashtra | Budget: 320 cr


### Debugging Exercise 2 — the missing parentheses
This runs but prints nothing:

```python
district = DistrictData("Patna", "Bihar", 160)
district.display          # author expected details to print
```
`district.display` (no `()`) refers to the method without running it. Fix: `district.display()`.

In [ ]:
district = DistrictData("Patna", "Bihar", 160)
district.display()   # parentheses make it actually run

District: Patna | State: Bihar | Budget: 160 cr


**Checkpoint — Guided Practice:** every class followed the same `__init__` + method shape; `display()` is a reusable habit; repetition makes classes routine.
**Key Takeaway:** Once you know the `__init__` + method shape, you can model almost any governance "thing".

---

## 7. Refactoring — From Dictionaries to a Class

**What it is:** **Refactor** = rewrite code cleaner *without changing what it does*. We take dictionary-based scheme code and turn it into a class.
**Why it matters:** This is exactly the clean-up you'll do to Mini Project 2 (MP2) for homework.

### The "before": procedural, dictionary-based

In [ ]:
# BEFORE — a dict + separate functions
scheme = {"name": "PM-KISAN", "coverage_percent": 92}

def show_scheme(scheme):
    print("Scheme:", scheme["name"], "| Coverage:", scheme["coverage_percent"], "%")

def meets_target(scheme):
    # National coverage target is 80%
    return scheme["coverage_percent"] >= 80

show_scheme(scheme)
print("Meets target?", meets_target(scheme))

Scheme: PM-KISAN | Coverage: 92 %
Meets target? True


### What's uncomfortable?
Every function must remember the exact keys; the data and functions float separately; you re-pass the dict everywhere.

### The "after": the same thing, as a class

In [ ]:
# AFTER — same behaviour, refactored into a class
class GovernmentScheme:
    def __init__(self, name, coverage_percent):
        self.name = name
        self.coverage_percent = coverage_percent

    def show(self):
        print("Scheme:", self.name, "| Coverage:", self.coverage_percent, "%")

    def meets_target(self):
        # National coverage target is 80%
        return self.coverage_percent >= 80

pmkisan = GovernmentScheme("PM-KISAN", 92)
pmkisan.show()
print("Meets target?", pmkisan.meets_target())

Scheme: PM-KISAN | Coverage: 92 %
Meets target? True


### Output discussion — what improved?
Identical output, cleaner code: data and behaviour travel together, `pmkisan.show()` reads like a sentence, the `"coverage_percent"` key isn't scattered around, and each new scheme is just `GovernmentScheme("PMAY", 74)`.

**Instructor Tip:** When the same dictionary is passed into several functions, that's a strong hint it wants to be a class.

### Prediction Question 3
Using the class above:

```python
pmay = GovernmentScheme("PMAY-G", 65)
print(pmay.meets_target())
```
**Answer:** `False` — `65 >= 80` is False.

**Checkpoint — Refactoring:** refactoring changes structure, not behaviour; dict + loose functions → one class; the win is readability and maintainability.
**Key Takeaway:** Repeated dict-passing is a signal to switch to a class.

---

## 8. Python Packages & `pip`

**What it is:** A **package/library** is pre-written code someone shared so you don't rewrite it. **Standard-library** packages (`math`, `random`, `datetime`) ship with Python; **third-party** ones (**NumPy**, **Pandas**) must be installed with `pip`.
**Why it matters:** All real policy analysis uses third-party libraries you install once.

In [ ]:
# Standard-library packages need no installation.
import math
import random

print("Square root of 144:", math.sqrt(144))
print("A random assembly constituency number 1-6:", random.randint(1, 6))

Square root of 144: 12.0
A random assembly constituency number 1-6: 5


In [ ]:
print(math.pi)

print(math.pow(5,2))

print(math.ceil(81/40))

print(math.floor(50999.95))

print(math.factorial(5))

3.141592653589793
25.0
3
50999
120


### Installing third-party packages with `pip`
`pip` downloads packages from **PyPI** and sets them up. Run it in a **terminal**, or in a Jupyter cell prefixed with `!`.

```bash
pip install numpy
pip install pandas
pip install -r requirements.txt   # install everything a project lists
```
A `requirements.txt` is your project's shopping list, one package per line:
```text
numpy
pandas
```
**Instructor Tip:** NumPy and Pandas are already installed here, so the `import` cells below just work.

In [ ]:
# Confirm NumPy and Pandas are available by importing and printing versions.
import numpy
import pandas

print("NumPy version:", numpy.__version__)
print("Pandas version:", pandas.__version__)

NumPy version: 2.0.2
Pandas version: 2.2.2


### Mini Exercise 7
Sort into standard-library vs third-party: `math`, `numpy`, `random`, `pandas`, `datetime`. Then write this session's `requirements.txt`.

**Solution**
- Standard library: `math`, `random`, `datetime`
- Third-party (`pip install`): `numpy`, `pandas`

```text
numpy
pandas
```
**Common Mistake:** Writing `pip install numpy` as a plain code line — it's a terminal command (use `!pip install numpy` inside Jupyter).
**Key Takeaway:** `pip install` fetches third-party tools; `requirements.txt` records them for reproducibility.

---

## 9. Virtual Environments — Why Professionals Isolate Projects

**What it is:** A **virtual environment** is an isolated folder with its own Python and its own installed packages, for one project.
**Why it matters:** Project A may need version 1.0 of a package and Project B version 2.0; isolating them avoids **dependency conflicts** — a private toolbox per project.

### The commands (run in a terminal)
```bash
python -m venv venv          # 1. create an environment named "venv"

# 2. activate it
venv\Scripts\activate        # Windows
source venv/bin/activate     # macOS / Linux

pip install numpy pandas     # 3. installs land ONLY in this environment
deactivate                   # 4. leave the environment
```
Once active, your prompt shows `(venv)`.

**Common Mistake:** Creating a venv but forgetting to **activate** it — if you don't see `(venv)`, you're not inside it.
**Key Takeaway:** One isolated environment per project keeps package versions from fighting.

---

## 10. Working Confidently in Jupyter Notebook

**What it is:** A Jupyter Notebook mixes **markdown cells** (text) and **code cells** (runnable Python). The **kernel** is the live Python brain that *remembers* every variable you've run until you restart it.
**Why it matters:** It's the standard workspace for data and policy analysis.

Press **Shift + Enter** to run a cell. **Restart & Run All** wipes memory and re-runs top to bottom — the honesty test for a reproducible notebook.

In [ ]:
# Proof the kernel remembers state across cells:
report_title = "District Development Index 2025"


In [ ]:
# This works ONLY because the previous cell ran and the kernel remembered the variable.
print("Analysing:", report_title)

Analysing: District Development Index 2025


### Jupyter best practices
Run top to bottom (do *Restart & Run All* often); one idea per cell; narrate with markdown; save with `Ctrl + S`; name variables meaningfully (`average_literacy`, not `x`).

### Mini Exercise 8
1. Which cell type writes a heading and explanation? 2. What does restarting the kernel do to your variables? 3. Why is "Restart & Run All" a good final check?

**Solution**
1. A **markdown** cell. 2. It **wipes all remembered variables** — you start blank and must re-run. 3. It proves the notebook runs **top to bottom** with no hidden state, so anyone can reproduce it.
**Key Takeaway:** A good notebook runs cleanly from top to bottom in a fresh kernel.

---

## 11. Git & GitHub — Version Control for Your Work

**What it is:** **Git** tracks the history of your files on your computer; **GitHub** is a website that stores Git projects online to back up and share.
**Why it matters:** It ends the `report_final_v2_FINAL.py` chaos and lets you roll back safely — essential for any shared policy project.

**Vocabulary:** *repository* (tracked folder), *commit* (saved snapshot + message), *push* (upload to GitHub), *README* (the project's front page).

### The essential commands (run in your project folder)
```bash
git init                                   # turn this folder into a repo (one-time)
git add .                                   # stage files for the next snapshot
git commit -m "Add Session 4 notebook and districts.csv"   # save a snapshot
git remote add origin https://github.com/your-username/Session-4.git   # one-time
git push origin main                        # upload commits to GitHub
```
**Instructor Tip:** Write commit messages a human can read months later — `"Add DistrictData class and Pandas section"`, not `"update"`.
**Common Mistake:** Forgetting `git add` before `git commit` — unstaged files aren't included. Flow is always **add → commit → push**.

### Mini Exercise 9
1. Order these: `git commit`, `git push`, `git add`. 2. Improve this commit message: `"update"` (for adding a Pandas section and districts.csv).

**Solution**
1. **`git add` → `git commit` → `git push`**. 2. e.g. `"Add Pandas section and districts.csv sample dataset"`.
**Key Takeaway:** Git is your save-point system, GitHub the cloud save — commit often with clear messages, then push.

---

## 12. Introduction to NumPy

**What it is:** **NumPy** gives you the **array** — a container built for fast maths on many numbers at once (**vectorization**), instead of looping element by element.
**Why it matters:** Census, budget, and survey datasets have thousands to millions of numbers; NumPy crunches them fast.

**Instructor Tip:** The universal import is `import numpy as np` — everyone writes `np`.

In [ ]:
import numpy as np

# A 1D array of state literacy rates (%).
literacy = np.array([87, 88, 77, 70, 95])

print(literacy)
print(type(literacy))

[87 88 77 70 95]
<class 'numpy.ndarray'>


### Output discussion
`np.array([...])` turned a list into a NumPy array — note it prints *without commas*. Its type is `numpy.ndarray`. Every array describes itself with `dtype`, `shape`, `ndim`, `size`.

In [ ]:
literacy = np.array([87, 88, 77, 70, 95])

print("dtype :", literacy.dtype)   # type of the numbers (e.g. int64)
print("shape :", literacy.shape)   # layout as a tuple: (5,) = 5 items in 1 row
print("ndim  :", literacy.ndim)    # number of dimensions
print("size  :", literacy.size)    # total number of elements

dtype : int64
shape : (5,)
ndim  : 1
size  : 5


### Output discussion — the four
`dtype=int64` (one number type for all elements — part of why it's fast); `shape=(5,)` (5 items, one dimension); `ndim=1`; `size=5`.

### The magic: vectorized maths on whole datasets

In [ ]:
allocations = np.array([100, 200, 300, 400])   # budget per district, in crore

print("Doubled grant:", allocations * 2)       # every element x2, no loop
print("Plus 50 cr   :", allocations + 50)      # add 50 to each

districts = np.array([1, 2, 3, 4])             # number of districts per allocation
print("Total needed :", allocations * districts)  # element-by-element

Doubled grant: [200 400 600 800]
Plus 50 cr   : [150 250 350 450]
Total needed : [ 100  400  900 1600]


### Output discussion — why it's a big deal
`allocations * 2` scaled *every* element in one step; `allocations * districts` multiplied the two arrays element by element. A plain Python list can't do this — `[100, 200] * 2` just *repeats* the list.

In [ ]:
# Proof: list vs array
plain_list = [100, 200, 300]
numpy_array = np.array([100, 200, 300])

print("list  * 2 ->", plain_list * 2)    # repeats the list
print("array * 2 ->", numpy_array * 2)   # doubles each number

list  * 2 -> [100, 200, 300, 100, 200, 300]
array * 2 -> [200 400 600]


**Common Mistake:** Expecting a Python list to do maths like an array — `my_list * 2` repeats, `my_list + 5` errors. Put numbers in an array first.

### Prediction Question 4
```python
import numpy as np
turnout = np.array([55, 62, 70])
print(turnout + 5)
```
**Answer:** `[60 67 75]` — 5 added to every element, printed array-style.

### Mini Exercise 10
Create `population = np.array([94, 46, 45, 58, 66])` (lakh). Print its `dtype`, `shape`, `ndim`, `size`. Then make a projected array with **+1 lakh** growth for every district.

In [ ]:
import numpy as np

population = np.array([94, 46, 45, 58, 66])

print("dtype:", population.dtype)
print("shape:", population.shape)
print("ndim :", population.ndim)
print("size :", population.size)

projected = population + 1   # vectorized: +1 lakh to every district
print("Projected:", projected)

dtype: int64
shape: (5,)
ndim : 1
size : 5
Projected: [95 47 46 59 67]


**Checkpoint — NumPy Basics:** arrays store numbers tightly and do whole-array maths; import as `np`; inspect with `dtype`/`shape`/`ndim`/`size`; arrays do element-wise maths, lists don't.
**Key Takeaway:** A NumPy array is a list built for maths — `array * 2` doubles every number, no loop.

---

## 13. NumPy Practice — Arrays, Shapes & Handy Constructors

**What it is:** 2D arrays (grids of rows × columns) and four constructors you'll use constantly: `zeros()`, `ones()`, `arange()`, `reshape()`, plus quick random integers.
**Why it matters:** Real data tables (states × years, districts × indicators) are 2D grids.

In [ ]:
import numpy as np

# A list of lists -> a 2D array: 2 rows x 3 columns.
# Rows = years, columns = three districts' scheme beneficiaries (in thousands).
grid = np.array([
    [12, 18, 9],
    [15, 20, 11]
])

print(grid)
print("shape:", grid.shape)   # (2, 3) = 2 rows, 3 columns
print("ndim :", grid.ndim)    # 2 dimensions
print("size :", grid.size)    # 6 elements total

[[12 18  9]
 [15 20 11]]
shape: (2, 3)
ndim : 2
size : 6


### Output discussion
Read `shape` as *(rows, columns)*: `(2, 3)`, `ndim=2`, `size=6`. This *(rows, columns)* habit pays off in Pandas next.

In [ ]:
import numpy as np

# zeros(): placeholder grid of 0.0
print("zeros:\n", np.zeros((2, 3)))

# ones(): grid of 1.0
print("\nones:\n", np.ones((2, 2)))

# arange(start, stop, step): like range(), but returns an array (stop is EXCLUSIVE)
print("\narange 0..9:", np.arange(0, 10))
print("arange step 2:", np.arange(0, 10, 2))

zeros:
 [[0. 0. 0.]
 [0. 0. 0.]]

ones:
 [[1. 1.]
 [1. 1.]]

arange 0..9: [0 1 2 3 4 5 6 7 8 9]
arange step 2: [0 2 4 6 8]


### `reshape()` — same numbers, new layout
`reshape()` rearranges an array into a new rows×columns layout without changing the numbers — the counts must match.

In [ ]:
import numpy as np

# Monthly budget spend for a year: 12 numbers in a flat row.
months = np.arange(1, 13)
print("Original:", months, "-> shape", months.shape)

# Reshape into 4 quarters x 3 months (4 * 3 = 12).
quarters = months.reshape(4, 3)
print("\nReshaped to 4x3 (quarters x months):\n", quarters)

Original: [ 1  2  3  4  5  6  7  8  9 10 11 12] -> shape (12,)

Reshaped to 4x3 (quarters x months):
 [[ 1  2  3]
 [ 4  5  6]
 [ 7  8  9]
 [10 11 12]]


### Output discussion
The twelve numbers didn't change — only their arrangement did. `reshape(4, 3)` works because `4 * 3 = 12`; `reshape(4, 4)` (needs 16) would error.

In [ ]:
import numpy as np

# Reproducible random generator (seed makes results repeatable).
rng = np.random.default_rng(seed=42)

# 5 random voter-turnout percentages (1-100, 100 exclusive).
turnout = rng.integers(1, 100, size=5)
print("Random turnout %:", turnout)

# A 2x3 grid of random beneficiary counts (1-10).
sample_grid = rng.integers(1, 10, size=(2, 3))
print("\nRandom 2x3 grid:\n", sample_grid)

Random turnout %: [ 9 77 65 44 43]

Random 2x3 grid:
 [[8 1 7]
 [2 1 5]]


**Instructor Tip:** A **seed** (`default_rng(seed=42)`) makes "random" numbers repeat identically every run — gold for teaching and reproducibility.

### Debugging Exercise 3 — the wrong alias
```python
import numpy as np
data = numpy.array([100, 200])   # NameError: 'numpy' is not defined
```
We imported it *as* `np`, so use `np.array(...)`, not `numpy.array(...)`.

In [ ]:
import numpy as np
data = np.array([100, 200])   # use the alias 'np'
print(data)

[100 200]


### Mini Exercise 11
1. Make a `3x3` array of **zeros**. 2. Make numbers **0–8** with `arange`, reshape to `3x3`. 3. Print that array's `shape` and `size`.

In [ ]:
import numpy as np

zeros_grid = np.zeros((3, 3))
print("Zeros grid:\n", zeros_grid)

count_grid = np.arange(0, 9).reshape(3, 3)   # 9 numbers fit a 3x3 grid
print("\nCount grid:\n", count_grid)

print("\nshape:", count_grid.shape, "| size:", count_grid.size)

Zeros grid:
 [[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]

Count grid:
 [[0 1 2]
 [3 4 5]
 [6 7 8]]

shape: (3, 3) | size: 9


### Mini Exercise 12
1. Make a `2x4` array of **ones**. 2. Make 6 **random integers** 1–50 (seed `7`), reshape to `2x3`, print it.

In [ ]:
import numpy as np

ones_grid = np.ones((2, 4))
print("Ones grid:\n", ones_grid)

rng = np.random.default_rng(seed=7)
random_numbers = rng.integers(1, 50, size=6)
print("\nRandom numbers:", random_numbers)

print("\nReshaped to 2x3:\n", random_numbers.reshape(2, 3))

Ones grid:
 [[1. 1. 1. 1.]
 [1. 1. 1. 1.]]

Random numbers: [47 31 34 44 29 39]

Reshaped to 2x3:
 [[47 31 34]
 [44 29 39]]


**Checkpoint — NumPy Practice:** 2D arrays are grids (read `shape` as rows, columns); `zeros`/`ones`/`arange` build arrays; `reshape` rearranges (counts must match); `default_rng(seed=...).integers()` makes reproducible randoms.
**Key Takeaway:** NumPy gives quick ways to build and reshape arrays — the raw material for numerical work.

---

## 14. Introduction to Pandas

**What it is:** A **DataFrame** is a spreadsheet inside Python — named columns, rows, and an index. A single column is a **Series**.
**Why it matters:** Every census table, scheme list, or budget sheet you analyse becomes a DataFrame.

**Instructor Tip:** The universal import is `import pandas as pd`.

In [ ]:
import pandas as pd

# Build a DataFrame by hand: each KEY is a column, each LIST is that column's values.
districts = pd.DataFrame({
    "district": ["Pune", "Lucknow", "Patna", "Ernakulam"],
    "state": ["Maharashtra", "Uttar Pradesh", "Bihar", "Kerala"],
    "literacy_rate": [87, 77, 70, 95]
})

districts   # last line of a cell displays automatically in Jupyter

,district,state,literacy_rate
0,Pune,Maharashtra,87
1,Lucknow,Uttar Pradesh,77
2,Patna,Bihar,70
3,Ernakulam,Kerala,95


### Output discussion — anatomy of a DataFrame
Three parts: **columns** (`district`, `state`, `literacy_rate`), **rows** (one per district), and the bold **index** (`0,1,2,3`) Pandas adds automatically.

In [ ]:
import pandas as pd

districts = pd.DataFrame({
    "district": ["Pune", "Lucknow", "Patna", "Ernakulam"],
    "state": ["Maharashtra", "Uttar Pradesh", "Bihar", "Kerala"],
    "literacy_rate": [87, 77, 70, 95]
})

# One column, selected by name in square brackets = a Series.
print(districts["district"])
print("\nType of one column:", type(districts["district"]))

0         Pune
1      Lucknow
2        Patna
3    Ernakulam
Name: district, dtype: object

Type of one column: <class 'pandas.core.series.Series'>


### Output discussion
`districts["district"]` pulled out one column — a **Series** — carrying its index along.

### Mini Exercise 13
Build a DataFrame `schemes` with columns `name`, `ministry`, `budget_crore` for **three** schemes. Display it, then print just the `name` column.

In [ ]:
import pandas as pd

schemes = pd.DataFrame({
    "name": ["PM-KISAN", "PMAY-G", "Ayushman Bharat"],
    "ministry": ["Agriculture", "Rural Development", "Health & Family Welfare"],
    "budget_crore": [63500, 54500, 7200]
})

print(schemes)
print("\nJust the names:")
print(schemes["name"])

              name                 ministry  budget_crore
0         PM-KISAN              Agriculture         63500
1           PMAY-G        Rural Development         54500
2  Ayushman Bharat  Health & Family Welfare          7200

Just the names:
0           PM-KISAN
1             PMAY-G
2    Ayushman Bharat
Name: name, dtype: object


**Common Mistake:** A typo in brackets — `districts["literacy"]` — raises `KeyError`. Use exact column names in quotes.
**Key Takeaway:** The DataFrame is the heart of data analysis — a spreadsheet you control with code.

---

## 15. Reading CSV Files

**What it is:** A **CSV** (Comma-Separated Values) is plain-text tabular data. `pd.read_csv()` loads it into a DataFrame in one line, then `head()`, `tail()`, `shape`, `columns`, `info()`, `describe()` explore it.
**Why it matters:** Census exports, ECI results, and budget tables all come as CSVs.

Our sample dataset — 10 districts with population, literacy, and budget:

| district | state | population_lakh | literacy_rate | budget_crore |
|----------|-------|-----------------|---------------|--------------|
| Pune | Maharashtra | 94 | 87 | 320 |
| Nagpur | Maharashtra | 46 | 88 | 210 |
| Lucknow | Uttar Pradesh | 45 | 77 | 180 |
| Patna | Bihar | 58 | 70 | 160 |
| Jaipur | Rajasthan | 66 | 76 | 240 |
| Indore | Madhya Pradesh | 33 | 80 | 150 |
| Ernakulam | Kerala | 33 | 95 | 130 |
| Cuttack | Odisha | 26 | 84 | 110 |
| Ludhiana | Punjab | 35 | 82 | 170 |
| Kamrup | Assam | 16 | 89 | 90 |

The next cell writes this to a real `districts.csv` so the notebook is self-contained.

In [ ]:
# Create districts.csv so this notebook is fully self-contained.
# (In a real project this file already exists and you start at read_csv below.)
csv_text = """district,state,population_lakh,literacy_rate,budget_crore
Pune,Maharashtra,94,87,320
Nagpur,Maharashtra,46,88,210
Lucknow,Uttar Pradesh,45,77,180
Patna,Bihar,58,70,160
Jaipur,Rajasthan,66,76,240
Indore,Madhya Pradesh,33,80,150
Ernakulam,Kerala,33,95,130
Cuttack,Odisha,26,84,110
Ludhiana,Punjab,35,82,170
Kamrup,Assam,16,89,90
"""

with open("districts.csv", "w") as f:
    f.write(csv_text)

print("districts.csv created!")

districts.csv created!


In [ ]:
import pandas as pd

# Read the CSV into a DataFrame — one line does it all.
districts = pd.read_csv("districts.csv")
districts

,district,state,population_lakh,literacy_rate,budget_crore
0,Pune,Maharashtra,94,87,320
1,Nagpur,Maharashtra,46,88,210
2,Lucknow,Uttar Pradesh,45,77,180
3,Patna,Bihar,58,70,160
4,Jaipur,Rajasthan,66,76,240
5,Indore,Madhya Pradesh,33,80,150
6,Ernakulam,Kerala,33,95,130
7,Cuttack,Odisha,26,84,110
8,Ludhiana,Punjab,35,82,170
9,Kamrup,Assam,16,89,90


### Output discussion
`pd.read_csv()` detected the columns from the header, inferred types (text for names, integers for numbers), and added an index — one line replaced manual file parsing.

### First five moves on any new dataset

In [ ]:
# head() -> the FIRST rows (default 5).
districts.head()

,district,state,population_lakh,literacy_rate,budget_crore
0,Pune,Maharashtra,94,87,320
1,Nagpur,Maharashtra,46,88,210
2,Lucknow,Uttar Pradesh,45,77,180
3,Patna,Bihar,58,70,160
4,Jaipur,Rajasthan,66,76,240


In [ ]:
# tail() -> the LAST rows (default 5).
districts.tail()

,district,state,population_lakh,literacy_rate,budget_crore
5,Indore,Madhya Pradesh,33,80,150
6,Ernakulam,Kerala,33,95,130
7,Cuttack,Odisha,26,84,110
8,Ludhiana,Punjab,35,82,170
9,Kamrup,Assam,16,89,90


In [ ]:
# Ask for a specific number of rows.
print("First 3 rows:")
print(districts.head(3))

First 3 rows:
  district          state  population_lakh  literacy_rate  budget_crore
0     Pune    Maharashtra               94             87           320
1   Nagpur    Maharashtra               46             88           210
2  Lucknow  Uttar Pradesh               45             77           180


In [ ]:
# shape -> (rows, columns), an ATTRIBUTE (no parentheses).
print("shape:", districts.shape)

# columns -> the column names.
print("columns:", list(districts.columns))

shape: (10, 5)
columns: ['district', 'state', 'population_lakh', 'literacy_rate', 'budget_crore']


In [ ]:
# info() -> column names, non-empty counts, and each type. Great for spotting missing data.
districts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   district         10 non-null     object
 1   state            10 non-null     object
 2   population_lakh  10 non-null     int64 
 3   literacy_rate    10 non-null     int64 
 4   budget_crore     10 non-null     int64 
dtypes: int64(3), object(2)
memory usage: 532.0+ bytes


In [ ]:
# describe() -> quick statistics for the NUMERIC columns only.
districts.describe()

,population_lakh,literacy_rate,budget_crore
count,10.000000,10.000000,10.00000
mean,45.200000,82.800000,176.00000
std,22.611698,7.345445,67.36303
min,16.000000,70.000000,90.00000
25%,33.000000,77.750000,135.00000
50%,40.000000,83.000000,165.00000
75%,55.000000,87.750000,202.50000
max,94.000000,95.000000,320.00000


### Output discussion — what each told us
`head()`/`tail()` preview rows; `shape` = `(10, 5)`; `columns` lists the five names; `info()` is a health check (non-null counts + types); `describe()` gives count/mean/min/max for numeric columns and skips text.

**Instructor Tip:** Make `head()`, `info()`, `describe()` a reflex the moment a dataset lands.

### Mini Exercise 14
Using `districts`: 1. show the **first 4 rows**; 2. print the **column names**; 3. print its **shape**.

In [ ]:
import pandas as pd
districts = pd.read_csv("districts.csv")

print("First 4 rows:")
print(districts.head(4))

print("\nColumns:", list(districts.columns))
print("Shape (rows, columns):", districts.shape)

First 4 rows:
  district          state  population_lakh  literacy_rate  budget_crore
0     Pune    Maharashtra               94             87           320
1   Nagpur    Maharashtra               46             88           210
2  Lucknow  Uttar Pradesh               45             77           180
3    Patna          Bihar               58             70           160

Columns: ['district', 'state', 'population_lakh', 'literacy_rate', 'budget_crore']
Shape (rows, columns): (10, 5)


### Prediction Question 5
`districts` has 10 rows and 5 columns. `print(districts.shape)` prints? **Answer:** `(10, 5)` — `shape` is `(rows, columns)`, an attribute (no parentheses).

**Checkpoint — Reading CSVs:** `pd.read_csv()` loads in one line; first moves are `head`/`tail`/`shape`/`columns`/`info`/`describe`; `shape`/`columns` are attributes, the rest are methods.
**Key Takeaway:** `read_csv` gets data in; `head`/`info`/`describe` tell you what you're holding.

---

## 16. Pandas Practice — Answering Real Questions

**What it is:** Use the DataFrame to answer everyday analyst questions — how many rows, averages, highest/lowest, and selecting the columns you care about.
**Why it matters:** "What's the average district literacy? Which district has the biggest budget?" is the daily work.

In [ ]:
import pandas as pd
districts = pd.read_csv("districts.csv")   # reload so this section stands alone

print("Rows:", districts.shape[0])       # first item of (rows, columns)
print("Columns:", districts.shape[1])    # second item

Rows: 10
Columns: 5


In [ ]:
# Statistics on a single column: pick the column, then call the method.
print("Average literacy:", districts["literacy_rate"].mean())
print("Highest literacy:", districts["literacy_rate"].max())
print("Lowest literacy :", districts["literacy_rate"].min())

Average literacy: 82.8
Highest literacy: 95
Lowest literacy : 70


### Output discussion
`districts["literacy_rate"]` selected the column (a Series); `.mean()`, `.max()`, `.min()` summarised all ten districts — Pandas did the loop for you.

### Selecting specific columns

In [ ]:
# ONE column -> a Series (single brackets)
just_names = districts["district"]

# MULTIPLE columns -> a smaller DataFrame (a LIST of names -> DOUBLE brackets)
name_and_budget = districts[["district", "budget_crore"]]

print("Just districts (a Series):")
print(just_names)
print("\nDistrict and budget (a DataFrame):")
print(name_and_budget)

Just districts (a Series):
0         Pune
1       Nagpur
2      Lucknow
3        Patna
4       Jaipur
5       Indore
6    Ernakulam
7      Cuttack
8     Ludhiana
9       Kamrup
Name: district, dtype: object

District and budget (a DataFrame):
    district  budget_crore
0       Pune           320
1     Nagpur           210
2    Lucknow           180
3      Patna           160
4     Jaipur           240
5     Indore           150
6  Ernakulam           130
7    Cuttack           110
8   Ludhiana           170
9     Kamrup            90


**Common Mistake:** `districts["district", "budget_crore"]` (single brackets) errors — use double brackets `districts[["district", "budget_crore"]]`.

### Mini Exercise 15
Using `districts`: 1. print the row count; 2. print the **average population_lakh**; 3. print a mini-table of only `district` and `state`.

In [ ]:
import pandas as pd
districts = pd.read_csv("districts.csv")

print("Number of districts:", districts.shape[0])
print("Average population (lakh):", districts["population_lakh"].mean())

print("\nDistrict and state:")
print(districts[["district", "state"]])

Number of districts: 10
Average population (lakh): 45.2

District and state:
    district           state
0       Pune     Maharashtra
1     Nagpur     Maharashtra
2    Lucknow   Uttar Pradesh
3      Patna           Bihar
4     Jaipur       Rajasthan
5     Indore  Madhya Pradesh
6  Ernakulam          Kerala
7    Cuttack          Odisha
8   Ludhiana          Punjab
9     Kamrup           Assam


### Challenge Problem 1 — combine NumPy *and* Pandas
From `districts`: 1. pull `literacy_rate` out as a **NumPy array**; 2. print its NumPy **average** and **highest**; 3. add **5 points** to every district (a projected literacy-drive gain) and print it.
Hint: `districts["literacy_rate"].to_numpy()` converts a column to a NumPy array.

In [ ]:
import numpy as np
import pandas as pd
districts = pd.read_csv("districts.csv")

literacy_array = districts["literacy_rate"].to_numpy()   # column -> NumPy array
print("Literacy as NumPy array:", literacy_array)

print("Average literacy:", literacy_array.mean())
print("Highest literacy:", literacy_array.max())

projected = literacy_array + 5   # vectorized literacy-drive gain
print("Projected after drive (+5):", projected)

Literacy as NumPy array: [87 88 77 70 76 80 95 84 82 89]
Average literacy: 82.8
Highest literacy: 95
Projected after drive (+5): [ 92  93  82  75  81  85 100  89  87  94]


**Why this works:** `to_numpy()` handed the column to NumPy, where `mean()`/`max()` summarised it and `+ 5` added the gain to every element — Pandas to *hold and select*, NumPy to *crunch*.
**Key Takeaway:** Select a column, then call a stat method — that answers most "average/most/least" questions in seconds.

---

## 17. Instructor Challenge — OOP Meets Pandas

**What it is:** The two halves of today shake hands — create `DistrictData` **objects** with a class, then load them into a **Pandas DataFrame** to analyse.
**Why it matters:** Real systems create objects as they run, then collect many into a DataFrame to summarise.

In [ ]:
import pandas as pd

# Step 1: the DistrictData class (same shape as Section 6).
class DistrictData:
    def __init__(self, name, state, budget_crore):
        self.name = name
        self.state = state
        self.budget_crore = budget_crore

    def display(self):
        print("District:", self.name, "| State:", self.state,
              "| Budget:", self.budget_crore, "cr")

# Step 2: create several DistrictData objects.
districts = [
    DistrictData("Pune", "Maharashtra", 320),
    DistrictData("Lucknow", "Uttar Pradesh", 180),
    DistrictData("Patna", "Bihar", 160),
    DistrictData("Jaipur", "Rajasthan", 240),
]

for one_district in districts:
    one_district.display()

District: Pune | State: Maharashtra | Budget: 320 cr
District: Lucknow | State: Uttar Pradesh | Budget: 180 cr
District: Patna | State: Bihar | Budget: 160 cr
District: Jaipur | State: Rajasthan | Budget: 240 cr


### From objects to a DataFrame
Turn each object into a dictionary of its attributes, then build a DataFrame from that list — each object becomes one row.

In [ ]:
import pandas as pd

district_rows = []
for one_district in districts:
    district_rows.append({
        "name": one_district.name,
        "state": one_district.state,
        "budget_crore": one_district.budget_crore
    })

district_df = pd.DataFrame(district_rows)
district_df

,name,state,budget_crore
0,Pune,Maharashtra,320
1,Lucknow,Uttar Pradesh,180
2,Patna,Bihar,160
3,Jaipur,Rajasthan,240


### Now analyse the objects with Pandas

In [ ]:
print("Number of districts:", district_df.shape[0])
print("Average budget (cr):", district_df["budget_crore"].mean())
print("Highest budget (cr):", district_df["budget_crore"].max())

print()
print(district_df.describe())

Number of districts: 4
Average budget (cr): 225.0
Highest budget (cr): 320

       budget_crore
count      4.000000
mean     225.000000
std       71.879529
min      160.000000
25%      175.000000
50%      210.000000
75%      260.000000
max      320.000000


### Output discussion
You *created* objects with OOP, then *collected and analysed* them with Pandas — objects in, DataFrame out, statistics computed. Model your "things" as classes, then load many into a DataFrame.

**Instructor Tip:** The bridge is turning each object into a dictionary of its attributes.
**Key Takeaway:** OOP builds and organises data as objects; Pandas analyses many at once.

---

## 18. GitHub Deliverable — What to Hand In

**Repository structure** — create a `Session-4/` folder:
```text
Session-4/
├── Session4_FollowAlong.ipynb   # this notebook
├── districts.csv                # the sample dataset
└── README.md                    # short project description
```

**README template:**
```markdown
# Session 4 — OOP Basics, Tooling & First Libraries

Follow-along notebook for Session 4 of the Python for Policy & Governance course.

## Contents
- `Session4_FollowAlong.ipynb` — the full lesson notebook
- `districts.csv` — sample district dataset used in the Pandas section

## Topics covered
- OOP: classes, objects, `__init__`, methods (GovernmentScheme, DistrictData, UnionBudget)
- Tooling: pip, virtual environments, Jupyter, Git & GitHub
- NumPy arrays and Pandas DataFrames on district/census/budget data

## How to run
1. Install requirements: `pip install numpy pandas`
2. Open in Jupyter and run all cells top to bottom.
```

**Commit messages — describe the change:**

| Weak (avoid) | Strong (use) |
|--------------|--------------|
| `update` | `Add Session 4 follow-along notebook` |
| `stuff` | `Add districts.csv sample dataset for Pandas practice` |
| `fixed it` | `Fix missing self in GovernmentScheme constructor` |
| `changes` | `Add README with setup and run instructions` |

**Key Takeaway:** A tidy repo (notebook + data + README) with clear commit messages is what "done" looks like.

---

## 19. Session Recap

**Object-Oriented Programming** — OOP bundles data + behaviour into an **object**; a **class** is a blueprint, an object is built with `ClassName()`; `__init__(self, ...)` sets up each object's data; `self` means "this object"; a **method** is behaviour called with `()`; refactor dicts + loose functions into a clean class.

**Professional Tooling** — **`pip install`** gets third-party libraries; `requirements.txt` lists them; **virtual environments** isolate a project's packages; **Jupyter**'s kernel remembers state (Restart & Run All is the honesty test); **Git/GitHub** version control with **add → commit → push**.

**NumPy** — the **array** does fast **vectorized** maths (`array * 2`); inspect with `dtype`/`shape`/`ndim`/`size`; build with `np.array`/`zeros`/`ones`/`arange` and rearrange with `reshape`.

**Pandas** — the **DataFrame** is a spreadsheet in Python; load with `pd.read_csv()`; explore with `head`/`tail`/`shape`/`columns`/`info`/`describe`; select with `df["col"]` / `df[["a","b"]]`; summarise with `.mean()`/`.max()`/`.min()`.

**Finale:** you loaded `DistrictData` **objects** into a **DataFrame** and analysed them — real governance analytics.

---

## 20. Homework

Do these in order and push everything to GitHub.

**1. Refactor Mini Project 2 (MP2) using classes.** Turn your dictionary-based MP2 into a class with a constructor and at least one method (like Section 7) — cleaner code, not new features.

**2. Install NumPy and Pandas yourself.** From a terminal: `pip install numpy pandas`, then import both and print their versions.

**3. Load a CSV into a DataFrame.** Use `districts.csv` (or any small public dataset — a scheme or census table). Load it with `pd.read_csv()` and run `head()`, `info()`, `describe()`. In a markdown cell, write **two sentences** on what you notice (e.g. which state has the highest literacy, the budget range).

**4. Push your notebook to GitHub.** Create a `Session-4` repo (notebook + `districts.csv` + `README.md`) using **add → commit → push** with clear messages.

**5. Write a professional README** following the Section 18 template.

**Instructor Tip:** Do the homework in a fresh notebook and finish with **Restart & Run All** — if it runs clean top to bottom, you're ready for Session 5.

**Stretch goal (optional):** Build a class of your choice — `ElectionResult`, `CensusRecord`, or `PolicyAnalysis` — create 4–5 objects, load them into a DataFrame, and print the average of one numeric attribute (turnout, literacy, or budget).

---

## End-of-Session Quiz (10 Questions)
Answer each before checking the key below.

**Q1. What is a *class* in Python?**
- A) A specific object built in memory
- B) A blueprint for creating objects
- C) A built-in data type like `int`
- D) A function that returns a list

**Q2. Which method runs automatically when you create an object?**
- A) `start()`  · B) `create()`  · C) `__init__()`  · D) `self()`

**Q3. What does `self` refer to inside a method?**
- A) The class blueprint itself
- B) The particular object the method is called on
- C) A required import
- D) The previous object created

**Q4. How do you correctly *run* a `describe` method on an object `scheme`?**
- A) `scheme.describe`  · B) `describe(scheme)`  · C) `scheme.describe()`  · D) `GovernmentScheme.describe`

**Q5. Which command installs a third-party package?**
- A) `python install numpy`  · B) `pip install numpy`  · C) `import numpy`  · D) `install numpy`

**Q6. Why do professionals use virtual environments?**
- A) To make code run faster
- B) To isolate each project's packages and avoid version conflicts
- C) To connect to GitHub
- D) To convert lists into arrays

**Q7. In Git, what is the correct everyday order?**
- A) push → commit → add  · B) commit → add → push  · C) add → commit → push  · D) add → push → commit

**Q8. What will `np.array([70, 76, 88]) * 2` produce?**
- A) `[70, 76, 88, 70, 76, 88]`  · B) `[140 152 176]`  · C) An error  · D) `[70, 76, 88, 2]`

**Q9. What does a DataFrame's `shape` attribute give you?**
- A) The column names  · B) A statistical summary  · C) A tuple of `(rows, columns)`  · D) The first five rows

**Q10. Which Pandas command loads a CSV file into a DataFrame?**
- A) `pd.open_csv()`  · B) `pd.read_csv()`  · C) `pd.load_csv()`  · D) `pd.csv()`

### Quiz Answer Key
1. **B** — a class is a blueprint; objects are built from it.
2. **C** — `__init__()` runs automatically at object creation.
3. **B** — `self` is the particular object the method acts on.
4. **C** — methods are actions; call them with `()`: `scheme.describe()`.
5. **B** — `pip install numpy` fetches a third-party package.
6. **B** — virtual environments isolate packages per project.
7. **C** — the flow is **add → commit → push**.
8. **B** — NumPy is vectorized, so it doubles each element → `[140 152 176]`.
9. **C** — `shape` returns a `(rows, columns)` tuple (an attribute, no parentheses).
10. **B** — `pd.read_csv()` reads a CSV into a DataFrame.

**Scoring:** 9–10 = flying · 7–8 = solid, review the misses · below 7 = re-run the relevant sections and redo their mini exercises. Well done for doing the work today.